# Notebook 01: Document Ingestion and Chunking

## Objective
Load the cleaned documents from Notebook 00, split their text into manageable overlapping chunks, preserve document metadata, validate the chunks, and save them for the retrieval stage.

## Input
- `data/processed/cleaned_documents.jsonl`

## Output
- `data/processed/chunks.jsonl`
- `data/quality_reports/chunk_quality_report.csv`

## Why chunking?
Large documents are difficult to retrieve and send to an LLM as a whole. Chunking creates smaller text units that can be searched independently while retaining source information.

In [8]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/secure-rag-platform-1")
INPUT_FILE = PROJECT_ROOT / "data/processed/cleaned_documents.jsonl"
OUTPUT_FILE = PROJECT_ROOT / "data/processed/chunks.jsonl"
REPORT_DIR = PROJECT_ROOT / "data/quality_reports"

documents_df = pd.read_json(INPUT_FILE, lines=True)

assert not documents_df.empty, "No documents loaded."
assert documents_df["cleaned_text"].notna().all(), "Missing document text."

print(f"Loaded {len(documents_df)} documents.")

Loaded 13 documents.


Recursive chunking

In [13]:
%pip install langchain-text-splitters


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=['\n\n','\n','','']
)

print('langChain recursive text is ready.')

langChain recursive text is ready.


In [21]:
chunk_records = []

metadata_columns = [
    "document_id",
    "version",
    "status",
    "classification",
    "source_path",
    "effective_date"
]

for _, row in documents_df.iterrows(): #iterrow:iterates through its rows
    text = row["cleaned_text"]
    chunks = text_splitter.split_text(text)

    for chunk_index, chunk in enumerate(chunks): #each chunk with position(enumerate)
        record = {
            column: row[column]
            for column in metadata_columns
            if column in documents_df.columns
        }
# Include document version in chunk_id to prevent duplicate IDs across versions.
        record.update({
            "chunk_id": (
    f"{row.get('document_id', 'doc')}_v" #
    f"{row.get('version', 'unknown')}_chunk_{chunk_index:04d}"#string 4 digits
),
            "chunk_index": chunk_index,
            "chunk_text": chunk
        })

        chunk_records.append(record)

print(f"Documents processed: {len(documents_df)}")
print(f"Total chunks created: {len(chunk_records)}")
print(f"Empty chunks: {sum(not r['chunk_text'].strip() for r in chunk_records)}")
print("\nFirst chunk preview:")
print(chunk_records[0]["chunk_text"][:300] if chunk_records else "No chunks created.")

Documents processed: 13
Total chunks created: 42
Empty chunks: 0

First chunk preview:
Northwind Technologies Pvt. Ltd. | HR-BEN-005 Employee Benefits v4.0 | Internal
Page 1
Employee Benefits
Northwind Technologies Pvt. Ltd. - Human Resources Policy
Document ID
Version
Effective Date
Status
HR-BEN-005
4.0
2025-07-01
Active
Owner
Approved By
Classification
Review Cycle
Head of Total Re


chunks quality validation

In [22]:
import pandas as pd

chunks_df = pd.DataFrame(chunk_records)

assert not chunks_df.empty, "No chunks were created."
assert chunks_df["chunk_text"].notna().all(), "Missing chunk text."
assert chunks_df["chunk_id"].is_unique, "Duplicate chunk IDs."

quality_report = {
    "source_documents": len(documents_df),
    "total_chunks": len(chunks_df),
    "empty_chunks": int(
        chunks_df["chunk_text"].apply(lambda text: not text.strip()).sum()
    ),
    "duplicate_chunk_ids": int(chunks_df["chunk_id"].duplicated().sum()),
    "minimum_chunk_length": int(chunks_df["chunk_text"].str.len().min()),
    "maximum_chunk_length": int(chunks_df["chunk_text"].str.len().max()),
    "average_chunk_length": round(
        chunks_df["chunk_text"].str.len().mean(), 2
    ),
}

display(pd.DataFrame([quality_report]))

,source_documents,total_chunks,empty_chunks,duplicate_chunk_ids,minimum_chunk_length,maximum_chunk_length,average_chunk_length
0,13,42,0,0,164,796,645.12


exporting files

In [29]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

chunks_df.to_json(
    OUTPUT_FILE,
    orient="records",
    lines=True,
    force_ascii=False
)

pd.DataFrame([quality_report]).to_csv(
    REPORT_DIR / "chunk_quality_report.csv",
    index=False
)

print(f"Chunks saved: {OUTPUT_FILE}")
print(f"Quality report saved: {REPORT_DIR / 'chunk_quality_report.csv'}")
print(f"Total chunks saved: {len(chunks_df)}")

Chunks saved: /content/secure-rag-platform-1/data/processed/chunks.jsonl
Quality report saved: /content/secure-rag-platform-1/data/quality_reports/chunk_quality_report.csv
Total chunks saved: 42


In [28]:
from google.colab import files

uploaded = files.upload()